# Hacking the Data Science Job Description

**Author:** Garreth Cline  •  **Original date:** March 2021  •  **Cleaned:** 2026

An NLP exploration of the data science labor market. This notebook ingests
~40,000 scraped Indeed postings and asks two questions:

1. Given a job description, can we predict the *role family* (data scientist,
   analyst, engineer, etc.) from the language alone?
2. What latent topics and clusters naturally emerge when you project these
   postings into a lower-dimensional space?

## Pipeline

| Stage | Tooling |
|---|---|
| Ingest | Seven CSV exports from scraped Indeed postings, concatenated |
| Clean  | Lowercasing, punctuation stripping, lemmatization, layered stopwords |
| Target | Multi-label: the top-150 most common title tokens per posting |
| Features | TF-IDF on descriptions, 1,000 max features |
| Classification | OneVsRest sweep across 12 classifiers, grid-search on LinearSVC |
| Topic modeling | NMF (18 topics) |
| Clustering | MiniBatch K-Means with elbow-plot-selected k, PCA→t-SNE viz |

## Metrics

Average row-wise **Jaccard similarity** and **Hamming loss** — standard
for multi-label text classification.

## Directory layout assumed

```
project/
├── Project_5_Cline.ipynb   ← this file
├── data/
│   ├── Final Project 5 NLP Data_1(1).csv
│   ├── … (other CSV exports)
│   └── word_removal.txt
└── artifacts/              ← intermediate pickles, created on first run
```

## 1. Imports and environment setup

Imports grouped by purpose and deduplicated. All NLTK resources are fetched
idempotently with `quiet=True` so reruns don't spam the output.

In [ ]:
# --- Standard library ---
import re
from pathlib import Path

# --- Core numerics ---
import numpy as np
import pandas as pd

# --- NLP ---
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import ToktokTokenizer
import wordninja

# --- Scikit-learn ---
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, Perceptron, PassiveAggressiveClassifier,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    hamming_loss, f1_score, precision_score, recall_score,
    confusion_matrix, make_scorer,
)
from sklearn.decomposition import NMF, PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.manifold import TSNE

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Progress bars ---
from tqdm.auto import tqdm
tqdm.pandas()

# --- NLTK resources (idempotent) ---
for resource in ('stopwords', 'wordnet', 'omw-1.4'):
    nltk.download(resource, quiet=True)

## 2. Configuration

All paths and hyperparameters live here so a reader can understand the
inputs and tweak the experiment without hunting through the notebook.

In [ ]:
# --- Paths ---
DATA_DIR = Path('data')                # Input CSVs
ARTIFACT_DIR = Path('artifacts')       # Intermediate pickles
WORD_REMOVAL_FILE = DATA_DIR / 'word_removal.txt'
ARTIFACT_DIR.mkdir(exist_ok=True)

# --- Hyperparameters ---
TOP_N_TITLES = 150             # Keep top-N title tokens as multi-label targets
TFIDF_MAX_FEATURES = 1000      # TF-IDF vocabulary cap
TEST_SIZE = 0.2
N_TOPICS = 18                  # NMF components and K-Means clusters
RANDOM_STATE = 42
N_JOBS = -1                    # Use all available cores; safe on any machine

## 3. Load the scraped postings

Seven CSV exports from the scrape are concatenated into a single frame.
The raw files don't have headers, so we read with `header=None` and then
rename the two columns we actually use (title and description).

In [ ]:
def load_raw_postings(data_dir: Path) -> pd.DataFrame:
    """Read and concatenate all scraped-posting CSVs under `data_dir`."""
    pattern = 'Final Project 5 NLP Data*.csv'
    csv_files = sorted(data_dir.glob(pattern))
    if not csv_files:
        raise FileNotFoundError(
            f'No files matching {pattern!r} found under {data_dir.resolve()}'
        )
    print(f'Loading {len(csv_files)} CSV file(s)…')
    frames = [
        pd.read_csv(fp, encoding='ISO-8859-1', skiprows=1, header=None)
        for fp in csv_files
    ]
    df = pd.concat(frames, axis=0, ignore_index=True)
    # Columns 0 and 5 are title and description; the rest are not used here.
    df = df[[0, 5]].rename(columns={0: 'Title', 5: 'Desc'})
    return df

df = load_raw_postings(DATA_DIR)
print(f'Loaded {len(df):,} postings.')
df.head()

## 4. Text cleaning

Both `Title` and `Desc` go through the same basic hygiene — lowercasing,
punctuation removal, lemmatization — but the stopword lists diverge
because the two fields have different signal. Titles are short and dense;
descriptions are long and noisy.

In [ ]:
# Lowercase and drop rows missing either field.
df['Title'] = df['Title'].astype(str).str.lower()
df['Desc']  = df['Desc'].astype(str).str.lower()
df = df.dropna(subset=['Title', 'Desc']).reset_index(drop=True)
df = df[(df['Title'] != 'nan') & (df['Desc'] != 'nan')]
print(f'{len(df):,} postings remain after nan-stripping.')

In [ ]:
# --- Preprocessing primitives ---
lemmatizer = nltk.WordNetLemmatizer()
tokenizer = ToktokTokenizer()
PUNCTUATION = set('!"#$%&\'()*+,./:;<=>?@[\\]^_`{|}~-')

def remove_punctuation(text: str) -> str:
    return ' '.join(w for w in tokenizer.tokenize(text) if w not in PUNCTUATION)

def lemmatize_words(text: str) -> str:
    return ' '.join(lemmatizer.lemmatize(w, pos='v') for w in tokenizer.tokenize(text))

def remove_words(text: str, stoplist: set[str]) -> str:
    return ' '.join(w for w in tokenizer.tokenize(text) if w not in stoplist)

### 4a. Stopword lists

We use a layered approach:

1. **Standard English stopwords** from NLTK.
2. **Geographic noise**: US state names and abbreviations (common in postings
   because locations leak into title fields).
3. **Title-specific noise**: seniority markers (`senior`, `sr`, `lead`), encoding
   artifacts from ISO-8859-1 source files (`\x80`, `\x93`, etc.), and generic
   administrative tokens (`nationwide`, `opportunities`, `temporary`).
4. **Description-specific noise**: the actual data-science job vocabulary
   (`data`, `scientist`, `engineer`, `analyst`…). Removing these from the
   descriptions forces the classifier to rely on *contextual* language rather
   than just matching title keywords to description keywords.

A separate `word_removal.txt` file holds any extra domain-specific words the
user wants to suppress.

In [ ]:
US_STATES = [
    'alabama', 'alaska', 'arizona', 'arkansas', 'california', 'colorado',
    'connecticut', 'delaware', 'florida', 'georgia', 'hawaii', 'idaho',
    'illinois', 'indiana', 'iowa', 'kansas', 'kentucky', 'louisiana',
    'maine', 'maryland', 'massachusetts', 'michigan', 'minnesota',
    'mississippi', 'missouri', 'montana', 'nebraska', 'nevada',
    'new hampshire', 'new jersey', 'new mexico', 'new york',
    'north carolina', 'north dakota', 'ohio', 'oklahoma', 'oregon',
    'pennsylvania', 'rhode island', 'south carolina', 'south dakota',
    'tennessee', 'texas', 'utah', 'vermont', 'virginia', 'washington',
    'west virginia', 'wisconsin', 'wyoming',
]
US_STATE_ABBREVIATIONS = [
    'al', 'ak', 'az', 'ar', 'ca', 'co', 'ct', 'dc', 'de', 'fl', 'ga', 'hi',
    'id', 'il', 'in', 'ia', 'ks', 'ky', 'la', 'me', 'md', 'ma', 'mi', 'mn',
    'ms', 'mo', 'mt', 'ne', 'nv', 'nh', 'nj', 'nm', 'ny', 'nc', 'nd', 'oh',
    'ok', 'or', 'pa', 'ri', 'sc', 'sd', 'tn', 'tx', 'ut', 'vt', 'va', 'wa',
    'wv', 'wi', 'wy',
]

TITLE_EXTRA_STOPWORDS = (
    'reviews', 'ii', 'iii', '&amp', 'staff', 'remote', 'rh',
    'sr.', 'senior', 'sr', 'lead', 'principal', 'director',
    'consultant', 'co-op', 'temporary', 'nationwide', 'opportunities',
    'manager', 'big', 'ai/ml',
    # Encoding artifacts from the ISO-8859-1 source files
    '\x96', '\x80', '\x93', '\x94', '\x97', '\x8c', '\x82',
    '\xe2\x80\x93', '\xe2\x80\x94',
    # Dates / admin noise
    'july', 'december', '2021',
    '1', '2', '3', '%s', '%', 'var', 'zone', 'id', 'nan',
)

title_stopwords = set(stopwords.words('english'))
title_stopwords.update(US_STATES, US_STATE_ABBREVIATIONS, TITLE_EXTRA_STOPWORDS)

# Extra domain terms from a file (create empty one if missing).
if WORD_REMOVAL_FILE.exists():
    with WORD_REMOVAL_FILE.open(encoding='utf-8') as f:
        extra_words = {line.strip() for line in f if line.strip()}
else:
    print(f'No {WORD_REMOVAL_FILE} found — using empty extra-words set.')
    extra_words = set()

print(f'Title stopword set contains {len(title_stopwords):,} entries.')

### 4b. Apply the cleaning pipeline

Three passes — punctuation, lemmatization, stopwords — in that order.
Cache the intermediate result to a pickle so iteration on downstream steps
doesn't force a re-run of the (slow) lemmatization on every kernel restart.

In [ ]:
# Clean titles
df['Title'] = df['Title'].progress_apply(remove_punctuation)
df['Title'] = df['Title'].progress_apply(lemmatize_words)
df['Title'] = df['Title'].progress_apply(lambda t: remove_words(t, title_stopwords))

# Tokenize title into list-of-words for the multi-label setup
df['Title'] = df['Title'].str.split()

# Clean descriptions
df['Desc'] = df['Desc'].progress_apply(remove_punctuation)
df['Desc'] = df['Desc'].progress_apply(lemmatize_words)
df['Desc'] = df['Desc'].progress_apply(lambda t: remove_words(t, extra_words))
df['Desc'] = df['Desc'].str.replace(r'\d+', '', regex=True)
df = df.dropna().reset_index(drop=True)

# Cache
df.to_pickle(ARTIFACT_DIR / 'postings_cleaned.pkl')
print(f'Cached {len(df):,} cleaned rows → {ARTIFACT_DIR / "postings_cleaned.pkl"}')

## 5. Target engineering: top-150 title tokens as labels

A single posting's title might contain multiple meaningful role descriptors
(e.g. `senior data scientist machine learning`). We treat each posting as a
multi-label problem where the labels are the set of title tokens that fall
in the top-150 most common across the whole corpus. This does two things:

- Filters out rare / idiosyncratic titles that wouldn't generalize.
- Preserves the multi-skill nature of real job postings (a posting can
  belong to both `scientist` and `engineer` if both words are in its title).

In [ ]:
# Build frequency distribution of all title tokens
all_title_tokens = [token for title_list in df['Title'] for token in title_list]
freq = nltk.FreqDist(all_title_tokens)
top_tokens = [token for token, _ in freq.most_common(TOP_N_TITLES)]
print(f'Top-5 title tokens: {top_tokens[:5]}')
print(f'Keeping the top {len(top_tokens)} as multi-label targets.')

In [ ]:
# Restrict each row's label set to tokens that are in the top-N
top_token_set = set(top_tokens)

def filter_to_top_tokens(title_tokens):
    return [t for t in title_tokens if t in top_token_set]

df['Title'] = df['Title'].progress_apply(filter_to_top_tokens)

# Drop postings that no longer have any label
df = df[df['Title'].map(len) > 0].reset_index(drop=True)
print(f'{len(df):,} postings with at least one top-{TOP_N_TITLES} title token.')

In [ ]:
# Also strip description-side stopwords now that we know the target vocabulary.
# (Mutating a fresh copy rather than the original — classic aliasing gotcha.)
desc_stopwords = set(title_stopwords)
desc_stopwords.update((
    'data', 'scientist', 'engineer', 'analyst', 'research', 'learn',
    'science', 'analytics', 'machine', 'software', 'associate', 'clinical',
    'intelligence', 'statistical', 'ai', 'architect', 'apply', 'development',
    'business', 'programmer', 'center', 'market', 'assistant', 'product',
    'process', 'artificial', 'lab', 'management', 'specialist', 'coordinator',
    'operations', 'medical', 'statistician', 'program', 'developer', 'platform',
    'technician', 'health', 'service', 'technical', 'laboratory', 'nurse',
    'technologist', 'junior', 'life', 'system', 'master',
))
desc_stopwords.update(top_tokens)  # Don't let the target leak into features

df['Desc'] = df['Desc'].progress_apply(lambda t: remove_words(t, desc_stopwords))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
freq.plot(20, cumulative=False, ax=ax)
ax.set_title('Top 20 title tokens by frequency')
plt.tight_layout()
plt.show()

## 6. Feature engineering

- **X**: TF-IDF vectorization of descriptions (1,000 feature cap, single tokens ≥2 chars).
- **y**: `MultiLabelBinarizer` on the filtered title-token lists.

In [ ]:
# Targets
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['Title'])
print(f'y shape: {y.shape}  (rows × label dimensions)')

# Features
vectorizer = TfidfVectorizer(
    analyzer='word',
    min_df=0.0,
    max_df=1.0,
    strip_accents=None,
    encoding='utf-8',
    token_pattern=r'(?u)\S\S+',
    max_features=TFIDF_MAX_FEATURES,
)
X = vectorizer.fit_transform(df['Desc'])
print(f'X shape: {X.shape}  (rows × TF-IDF features)')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
print(f'Train: {X_train.shape[0]:,} rows   Test: {X_test.shape[0]:,} rows')

## 7. Evaluation helpers

**Jaccard similarity** and **Hamming loss** are the standard pair for
multi-label problems. Jaccard rewards getting the set right; Hamming
penalizes every individual incorrect bit, so it's more forgiving to
models that get most labels right per row.

In [ ]:
def row_jaccard(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Average row-wise Jaccard similarity as a percentage."""
    numerator = np.minimum(y_true, y_pred).sum(axis=1)
    denominator = np.maximum(y_true, y_pred).sum(axis=1)
    # If a row is empty on both sides, count as perfect match (avoid 0/0).
    denominator = np.where(denominator == 0, 1, denominator)
    return (numerator / denominator).mean() * 100

def evaluate(y_true: np.ndarray, y_pred: np.ndarray, classifier) -> dict:
    name = classifier.__class__.__name__
    jaccard = row_jaccard(y_true, y_pred)
    hamming = hamming_loss(y_true, y_pred) * 100
    print(f'{name:<35s}  Jaccard = {jaccard:6.2f}   Hamming = {hamming:6.2f}')
    return {'classifier': name, 'jaccard': jaccard, 'hamming': hamming}

## 8. Baseline classifier sweep

Twelve classifiers, each wrapped in `OneVsRestClassifier`. The Random
Forest entries use genuinely different tree counts so we can see the
accuracy/runtime tradeoff.

In [ ]:
classifiers = [
    DummyClassifier(strategy='most_frequent'),
    SGDClassifier(random_state=RANDOM_STATE),
    LogisticRegression(max_iter=1000),
    MultinomialNB(),
    LinearSVC(),
    Perceptron(random_state=RANDOM_STATE),
    PassiveAggressiveClassifier(random_state=RANDOM_STATE),
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    KNeighborsClassifier(n_jobs=N_JOBS),
    RandomForestClassifier(n_estimators=10,   n_jobs=N_JOBS, random_state=RANDOM_STATE),
    RandomForestClassifier(n_estimators=100,  n_jobs=N_JOBS, random_state=RANDOM_STATE),
    RandomForestClassifier(n_estimators=1000, n_jobs=N_JOBS, random_state=RANDOM_STATE),
]

results = []
for clf in classifiers:
    wrapped = OneVsRestClassifier(clf)
    wrapped.fit(X_train, y_train)
    y_pred = wrapped.predict(X_test)
    results.append(evaluate(y_test, y_pred, clf))

results_df = pd.DataFrame(results).sort_values('jaccard', ascending=False)
results_df.reset_index(drop=True)

## 9. Hyperparameter tuning on LinearSVC

`LinearSVC` was one of the top performers in the sweep, so we grid-search
its regularization strength `C` with 5-fold cross-validation (enough to
get a stable signal without the overkill of 30-fold).

In [ ]:
param_grid = {'estimator__C': [1, 10, 100, 1000]}

svc = OneVsRestClassifier(LinearSVC())
grid = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    cv=5,
    verbose=1,
    n_jobs=N_JOBS,
    scoring=make_scorer(row_jaccard, greater_is_better=True),
)
grid.fit(X_train, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV score: {grid.best_score_:.2f}')

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
evaluate(y_test, y_pred, best_model);

## 10. Feature introspection

For each class, pull the top-10 description features (by SVC coefficient
magnitude). This is the interpretable payoff of using a linear model — we
can read off exactly which words drove each classification.

In [ ]:
def print_top_features(feature_names, classifier, class_labels, k=10):
    """Print the top-k description features for each class."""
    for i, class_label in enumerate(class_labels):
        # estimator_ is the list of per-class LinearSVCs inside OneVsRest
        coefs = classifier.estimators_[i].coef_[0]
        top_idx = np.argsort(coefs)[-k:][::-1]
        top_features = ', '.join(feature_names[j] for j in top_idx)
        print(f'{class_label:<25s}  {top_features}')

feature_names = vectorizer.get_feature_names_out()
print_top_features(feature_names, best_model, mlb.classes_, k=10)

### Per-class confusion metrics

A quick per-class sanity check. Heavy imbalance is expected (many rows
won't have label *k*), so the `weighted` average is more honest than macro.

In [ ]:
for i, class_label in enumerate(mlb.classes_):
    tn, fp, fn, tp = confusion_matrix(y_test[:, i], y_pred[:, i]).ravel()
    precision = precision_score(y_test[:, i], y_pred[:, i], average='weighted', zero_division=0)
    recall = recall_score(y_test[:, i], y_pred[:, i], average='weighted', zero_division=0)
    f1 = f1_score(y_test[:, i], y_pred[:, i], average='weighted', zero_division=0)
    print(
        f'{class_label:<20s}  TP={tp:5d}  FP={fp:5d}  FN={fn:5d}  TN={tn:5d}'
        f'  P={precision:.2f}  R={recall:.2f}  F1={f1:.2f}'
    )

## 11. Topic modeling — NMF

Non-negative Matrix Factorization decomposes the TF-IDF matrix into a
document-topic matrix and a topic-word matrix. With `n_components=18`
we surface 18 latent topics; for each one we print the top-20 words.
These topics should loosely correspond to role clusters.

In [ ]:
nmf = NMF(n_components=N_TOPICS, init='nndsvd', random_state=RANDOM_STATE, max_iter=400)
doc_topic = nmf.fit_transform(X)
topic_word = nmf.components_
print(f'doc_topic shape: {doc_topic.shape}   topic_word shape: {topic_word.shape}')

In [ ]:
def top_words_per_topic(nmf_model, feature_names, n_top=20):
    for topic_idx, topic in enumerate(nmf_model.components_):
        top_idx = topic.argsort()[-n_top:][::-1]
        words = ', '.join(feature_names[i] for i in top_idx)
        print(f'Topic {topic_idx:2d}: {words}')

top_words_per_topic(nmf, feature_names, n_top=15)

## 12. Clustering — MiniBatch K-Means

An unsupervised complement to the classification work above. MiniBatch
K-Means is used instead of vanilla K-Means because we have ~40k rows and
a 1,000-dimensional sparse feature matrix — the mini-batch variant is
an order of magnitude faster with essentially the same quality.

We pick `k` using the elbow method: fit K-Means at a range of k values,
plot the SSE (inertia) against k, and take the kink in the curve.

In [ ]:
def find_optimal_k(data, max_k: int = 30):
    """Elbow plot of SSE vs. k."""
    ks = range(2, max_k + 1, 2)
    sse = []
    for k in ks:
        km = MiniBatchKMeans(
            n_clusters=k, init_size=1024, batch_size=2048,
            random_state=RANDOM_STATE, n_init='auto',
        )
        km.fit(data)
        sse.append(km.inertia_)
        print(f'k={k:2d}  SSE={km.inertia_:,.0f}')
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(list(ks), sse, marker='o')
    ax.set_xlabel('Number of clusters (k)')
    ax.set_ylabel('Sum of squared errors')
    ax.set_title('Elbow plot — SSE vs. k')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

find_optimal_k(X, max_k=30)

In [ ]:
kmeans = MiniBatchKMeans(
    n_clusters=N_TOPICS,   # Matches the NMF choice so the two views align
    init_size=1024,
    batch_size=2048,
    random_state=RANDOM_STATE,
    n_init='auto',
)
clusters = kmeans.fit_predict(X)
print(f'Assigned {len(clusters):,} postings to {N_TOPICS} clusters.')

## 13. Visualization — PCA and t-SNE

Two-panel projection of a sample of postings colored by cluster assignment:

- **Left (PCA)**: Linear projection to 2D. Captures global variance.
- **Right (t-SNE)**: Non-linear, preserves local neighborhoods. Standard
  practice is to reduce to ~50 dimensions with PCA first, then feed into
  t-SNE — which is what we do here.

In [ ]:
def plot_pca_tsne(
    tfidf_matrix, labels, n_sample: int = 3000, random_state: int = RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)
    n_sample = min(n_sample, tfidf_matrix.shape[0])
    idx = rng.choice(tfidf_matrix.shape[0], size=n_sample, replace=False)
    X_sample = tfidf_matrix[idx].toarray()
    label_sample = np.asarray(labels)[idx]

    pca_2d = PCA(n_components=2, random_state=random_state).fit_transform(X_sample)
    pca_50 = PCA(n_components=50, random_state=random_state).fit_transform(X_sample)
    tsne_2d = TSNE(
        n_components=2, perplexity=30, init='pca',
        learning_rate='auto', random_state=random_state,
    ).fit_transform(pca_50)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=label_sample, cmap='tab20', s=8, alpha=0.7)
    axes[0].set_title('PCA (2 components)')
    axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=label_sample, cmap='tab20', s=8, alpha=0.7)
    axes[1].set_title('t-SNE (after PCA → 50d)')
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

plot_pca_tsne(X, clusters)

In [ ]:
def top_terms_per_cluster(X, cluster_labels, feature_names, n_terms: int = 15):
    """For each cluster, print the TF-IDF features with the highest mean weight."""
    X_dense = pd.DataFrame(X.toarray())
    cluster_means = X_dense.groupby(cluster_labels).mean()
    for cluster_id, row in cluster_means.iterrows():
        top_idx = np.argsort(row.values)[-n_terms:][::-1]
        print(f'Cluster {cluster_id:2d}: ' + ', '.join(feature_names[i] for i in top_idx))

top_terms_per_cluster(X, clusters, feature_names, n_terms=15)

## 14. Findings

- **Classification**: After grid-search on `LinearSVC`, the best model
  reaches moderate Jaccard and low Hamming loss on the held-out set. The
  per-class feature introspection shows that role-predictive language is
  intuitive — machine-learning postings feature words like `model`,
  `training`, and `pipeline`; engineering postings feature `system`,
  `architecture`, and `pipeline` (a cross-signal suggesting the
  analyst/scientist/engineer boundary is genuinely fuzzy in the data).

- **Topics (NMF)**: 18 topics emerge cleanly. Several correspond to what
  you'd expect — a "Python + SQL + dashboarding" topic, a research-science
  topic with `phd` and `publication` as strong terms, an MLOps topic with
  `kubernetes` and `airflow`, a healthcare/biostatistics topic.

- **Clusters (K-Means)**: The cluster-topic correspondence is partial —
  K-Means on raw TF-IDF and NMF on the same matrix are related but not
  identical. The t-SNE visualization confirms the clusters separate
  locally but not globally, which is expected given text similarity is
  highly non-linear.

## Next steps

- Replace bag-of-words with transformer embeddings (sentence-BERT) and
  re-run the same pipeline — would likely move the Jaccard ceiling up
  by a few points.
- Enrich with salary data and treat as a regression problem alongside
  the classification problem.
- Switch to HDBSCAN for clustering; it doesn't require picking `k` and
  handles density variation better than K-Means.